# 🔴 Notebook 1 — Pipeline de Clasificación + Explicación con LLM
> **El modelo predice, el LLM explica**

En este notebook vamos a:
1. Entrenar un clasificador de churn con `scikit-learn`
2. Pasar las predicciones + features al LLM (Gemini via LangChain)
3. Generar explicaciones en lenguaje natural para cada predicción

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [1]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 13.4 MB/s eta 0:00:00


In [33]:
!pip install -q transformers accelerate torch

## 2. Configuración de API Key

In [23]:
import os
from dotenv import load_dotenv
from google.colab import userdata

load_dotenv()

# Opción A: desde .env
# GOOGLE_API_KEY=tu_clave en archivo .env

# Opción B: directamente (solo para desarrollo)
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
print("API Key cargada:", "✅" if GOOGLE_API_KEY else "❌ No encontrada")

API Key cargada: ✅


## 3. Crear dataset sintético de Churn

In [24]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
n = 500

df = pd.DataFrame({
    "antiguedad_meses": np.random.randint(1, 60, n),
    "num_productos": np.random.randint(1, 5, n),
    "saldo_promedio": np.random.uniform(0, 50000, n).round(2),
    "llamadas_soporte": np.random.randint(0, 10, n),
    "uso_app_mensual": np.random.randint(0, 30, n),
    "plan": np.random.choice(["básico", "estándar", "premium"], n),
})

# Regla sintética de churn
df["churn"] = (
    (df["llamadas_soporte"] > 6) |
    (df["uso_app_mensual"] < 3) |
    ((df["saldo_promedio"] < 500) & (df["antiguedad_meses"] < 6))
).astype(int)

print(f"Dataset: {df.shape}")
print(f"Churn rate: {df['churn'].mean():.1%}")
df.head()

Dataset: (500, 7)
Churn rate: 38.8%


,antiguedad_meses,num_productos,saldo_promedio,llamadas_soporte,uso_app_mensual,plan,churn
0,39,2,27771.59,1,10,premium,0
1,52,4,38449.37,2,9,premium,0
2,29,3,47238.29,4,29,premium,0
3,15,3,42482.37,7,17,premium,1
4,43,2,12367.41,9,17,básico,1


## 4. Entrenar el clasificador ML

In [25]:
le = LabelEncoder()
df["plan_encoded"] = le.fit_transform(df["plan"])

features = ["antiguedad_meses", "num_productos", "saldo_promedio",
            "llamadas_soporte", "uso_app_mensual", "plan_encoded"]

X = df[features]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

              precision    recall  f1-score   support

    No Churn       1.00      1.00      1.00        62
       Churn       1.00      1.00      1.00        38

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



## 5. Configurar LangChain + Gemini

In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)


# Prompt de explicación
prompt = ChatPromptTemplate.from_template("""
Eres un analista de retención de clientes. Dado el perfil de un cliente y la predicción del modelo,
genera una explicación clara y accionable en español.

**Perfil del cliente:**
- Antigüedad: {antiguedad_meses} meses
- Número de productos contratados: {num_productos}
- Saldo promedio: ${saldo_promedio:,.0f}
- Llamadas al soporte (último mes): {llamadas_soporte}
- Uso de la app (días/mes): {uso_app_mensual}
- Plan: {plan}

**Predicción del modelo:** {prediccion} (probabilidad de churn: {probabilidad:.0%})

Responde con:
1. **Diagnóstico** (2 oraciones): Por qué el modelo tomó esta decisión
2. **Señales de alerta**: Lista los factores de riesgo principales
3. **Acción recomendada**: Qué debería hacer el equipo de retención
""")

chain = prompt | llm | StrOutputParser()

print("✅ Chain de explicación configurada")

✅ Chain de explicación configurada


## 6. Generar explicaciones para los primeros casos de churn

In [35]:
# Tomamos los primeros 3 casos predichos como churn
X_test_df = X_test.copy()
X_test_df["prediccion_num"] = y_pred
X_test_df["probabilidad"] = y_prob
X_test_df["plan"] = df.loc[X_test.index, "plan"].values

casos_churn = X_test_df[X_test_df["prediccion_num"] == 1].head(3)

for i, (idx, row) in enumerate(casos_churn.iterrows()):
    print(f"\n{'='*60}")
    print(f"📋 CLIENTE #{i+1} (ID: {idx})")
    print(f"{'='*60}")

    explicacion = chain.invoke({
        "antiguedad_meses": int(row["antiguedad_meses"]),
        "num_productos": int(row["num_productos"]),
        "saldo_promedio": float(row["saldo_promedio"]),
        "llamadas_soporte": int(row["llamadas_soporte"]),
        "uso_app_mensual": int(row["uso_app_mensual"]),
        "plan": row["plan"],
        "prediccion": "CHURN" if row["prediccion_num"] == 1 else "NO CHURN",
        "probabilidad": float(row["probabilidad"])
    })

    print(explicacion)


📋 CLIENTE #1 (ID: 73)
Aquí tienes el análisis de retención para el perfil del cliente:

---

1.  **Diagnóstico:**
    El modelo predice un churn con una probabilidad extremadamente alta (97%) debido a una severa y reciente escalada de frustración del cliente, evidenciada por un número excesivo de llamadas al soporte en el último mes. A pesar de su alta interacción con la app y múltiples productos, estas interacciones negativas sugieren problemas críticos no resueltos que están impulsando su salida.

2.  **Señales de alerta:**
    *   **Alta Frecuencia de Llamadas al Soporte:** 8 llamadas en el último mes es el indicador más alarmante, señalando una frustración severa o problemas persistentes y no resueltos.
    *   **Contradicción Uso-Problema:** El cliente utiliza activamente la app (20 días/mes) y tiene 4 productos, lo que sugiere un deseo de interactuar con el servicio, pero las llamadas al soporte indican que no puede hacerlo de manera satisfactoria.
    *   **Riesgo de Pérdida de

## 7. Pipeline completo como función reutilizable

In [36]:
def predecir_y_explicar(cliente: dict) -> dict:
    """
    Pipeline completo: recibe datos de un cliente,
    predice churn y genera explicación con LLM.
    """
    # 1. Preparar features
    plan_enc = le.transform([cliente["plan"]])[0]
    features_input = [[
        cliente["antiguedad_meses"],
        cliente["num_productos"],
        cliente["saldo_promedio"],
        cliente["llamadas_soporte"],
        cliente["uso_app_mensual"],
        plan_enc
    ]]

    # 2. Predicción ML
    pred = clf.predict(features_input)[0]
    prob = clf.predict_proba(features_input)[0][1]

    # 3. Explicación LLM
    explicacion = chain.invoke({
        **cliente,
        "prediccion": "CHURN" if pred == 1 else "NO CHURN",
        "probabilidad": prob
    })

    return {
        "prediccion": "CHURN" if pred == 1 else "NO CHURN",
        "probabilidad_churn": f"{prob:.1%}",
        "explicacion": explicacion
    }

# --- Prueba con cliente nuevo ---
nuevo_cliente = {
    "antiguedad_meses": 3,
    "num_productos": 1,
    "saldo_promedio": 200,
    "llamadas_soporte": 8,
    "uso_app_mensual": 1,
    "plan": "básico"
}

resultado = predecir_y_explicar(nuevo_cliente)
print(f"Predicción: {resultado['prediccion']}")
print(f"Probabilidad: {resultado['probabilidad_churn']}")
print(f"\n{resultado['explicacion']}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Predicción: CHURN
Probabilidad: 94.0%

Aquí tienes el análisis:

---

**1. Diagnóstico**
El modelo predice un riesgo de churn extremadamente alto (94%) debido a una clara señal de insatisfacción y frustración por parte del cliente. Esto se evidencia principalmente por el elevado número de interacciones con soporte técnico y el bajo uso de los canales digitales, a pesar de su corta antigüedad.

**2. Señales de alerta**
*   **Elevado número de llamadas al soporte (8 en el último mes):** Indica problemas recurrentes, dificultad para resolver incidencias o una experiencia de usuario muy deficiente. Es el factor de riesgo más crítico.
*   **Bajo uso de la aplicación (1 día/mes):** Sugiere que el cliente no encuentra valor en la app, no sabe usarla, o sus problemas lo obligan a recurrir al soporte telefónico.
*   **Corta antigüedad (3 meses):** Los clientes nuevos son más propensos al churn si su experiencia inicial es negativa.
*   **Baja vinculación (1 producto, saldo bajo):** Menor "costo

## 🎯 Conclusiones

- El modelo ML es **rápido y escalable** para hacer predicciones masivas
- El LLM agrega una **capa de interpretabilidad** en lenguaje natural
- Este patrón es muy útil en producción para **equipos no técnicos** que necesitan actuar sobre predicciones

**Patrón clave:** `ML Model → Predicción + Features → LLM → Explicación accionable`